In [1]:
# --- Cell A: ML-in-SRE recon. Read-only, no downloads, no MCL budget. ---
library(reticulate)

hdr <- function(x) { cat("\n== ", x, " ==\n", sep = ""); flush.console() }
say <- function(k, v) { cat(sprintf("  %-22s %s\n", k, paste(v, collapse = " ")));
                        flush.console() }
tryv <- function(expr) tryCatch(expr, error = function(e)
                        paste("ERR:", conditionMessage(e)))

hdr("1. which python is reticulate using")
say("python",  tryv(py_config()$python))
say("version", tryv(py_config()$version))

hdr("2. module availability")
for (m in c("metacontentlibraryapi", "fbri", "fbri.package_managers",
            "fbri.package_managers.huggingface",
            "transformers", "torch", "sentence_transformers")) {
  say(m, tryv(if (py_module_available(m)) "OK" else "NOT AVAILABLE"))
}

hdr("3. the huggingface helper")
hf <- tryv(import("fbri.package_managers.huggingface"))
if (is.character(hf)) {
  say("import", hf)                       # the error text is the finding
} else {
  ins <- import("inspect")
  say("module file",      tryv(hf$`__file__`))
  say("hf_download_repo", tryv(py_str(ins$signature(hf$hf_download_repo))))
  say("hf_download_file", tryv(py_str(ins$signature(hf$hf_download_file))))
  say("exports", tryv(paste(grep("^_", py_list_attributes(hf), invert = TRUE,
                                 value = TRUE), collapse = ", ")))
}

hdr("4. hardware")
if (isTRUE(tryv(py_module_available("torch")))) {
  torch <- import("torch")
  say("torch version",  tryv(torch$`__version__`))
  say("cuda available", tryv(torch$cuda$is_available()))
} else say("torch", "not available")

hdr("5. filesystem")
say("home", path.expand("~"))
hfdir <- file.path(path.expand("~"), "huggingface")
say("~/huggingface exists", dir.exists(hfdir))
if (dir.exists(hfdir))
  say("subdirs", head(list.dirs(hfdir, recursive = TRUE, full.names = FALSE), 12))
say("free space", tryv(system(paste("df -h", shQuote(path.expand("~")), "| tail -1"),
                              intern = TRUE)))

hdr("6. SOURCE of hf_download_repo  <- the important one")
if (!is.character(hf)) {
  for (fn in c("hf_download_repo", "hf_download_file")) {
    src <- tryv(import("inspect")$getsource(py_get_attr(hf, fn)))
    cat("\n--- ", fn, " ---\n", sep = "")
    cat(paste(head(strsplit(as.character(src), "\n")[[1]], 45), collapse = "\n"), "\n")
    flush.console()
  }
}

[NOTICE] 2 output(s) filtered out

In [2]:
library(reticulate)
hf <- import("fbri.package_managers.huggingface")

cat("HF_ENDPOINT: ", tryCatch(hf$HF_ENDPOINT, error = function(e) "<unreadable>"),
    "\n\n", sep = "")
flush.console()

probe <- function(repo, note) {
  res <- tryCatch({
    f <- unlist(hf$hf_list_files(repo, "main"))
    sprintf("OK  %d files  |  %s", length(f), paste(utils::head(f, 4), collapse = ", "))
  }, error = function(e) {
    paste("ERR ", substr(gsub("[\r\n]+", " ", conditionMessage(e)), 1, 200))
  })
  cat(repo, "  ", note, "\n   -> ", res, "\n\n", sep = "")
  flush.console()
}

probe("facebook/mbart-large-50-many-to-many-mmt", "[approved, id printed verbatim by Meta]")
probe("facebook/nllb-200-distilled-600M",         "[approved, org prefix GUESSED]")
probe("sentence-transformers/all-MiniLM-L6-v2",   "[approved, org prefix GUESSED]")
probe("facebook/mbart-large-50-many-to-many-XXX", "[typo: no such repo]")
probe("gpt2",                                     "[real model, NOT on Meta's list]")

[NOTICE] 1 output(s) filtered out

In [3]:
# --- Cell B2: resolve the remaining repo ids. DOWNLOADS NOTHING. ---
library(reticulate)
hf <- import("fbri.package_managers.huggingface")

try_repo <- function(repo) {
  tryCatch({
    sprintf("OK %4d files", length(unlist(hf$hf_list_files(repo, "main"))))
  }, error = function(e) {
    code <- regmatches(conditionMessage(e),
                       regexpr("[0-9]{3} (Client|Server) Error", conditionMessage(e)))
    if (!length(code)) "ERR (other)" else sprintf("ERR %s", code)
  })
}

targets <- list(
  list("#1  Facebook NLLB-200-3.3B",      c("facebook/nllb-200-3.3B")),
  list("#3  Google T5",                   c("t5-base", "google-t5/t5-base")),
  list("#4  Google T5-small",             c("t5-small", "google-t5/t5-small")),
  list("#5  Google BERT base uncased",    c("bert-base-uncased", "google-bert/bert-base-uncased")),
  list("#6  Facebook mBART-50",           c("facebook/mbart-large-50")),
  list("#9  DistilBERT SST-2",            c("distilbert-base-uncased-finetuned-sst-2-english",
                                            "distilbert/distilbert-base-uncased-finetuned-sst-2-english")),
  list("#10 XLM-RoBERTa large",           c("xlm-roberta-large", "FacebookAI/xlm-roberta-large")),
  list("#11 DeBERTaV3",                   c("microsoft/deberta-v3-base", "microsoft/deberta-v3-large")),
  list("#12 mDeBERTa v3 multilingual",    c("microsoft/mdeberta-v3-base")),
  list("#13 paraphrase-multilingual",     c("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"))
)

for (t in targets) {
  cat(t[[1]], "\n", sep = "")
  for (r in t[[2]]) { cat("      ", try_repo(r), "  ", r, "\n", sep = ""); flush.console() }
}
cat("\ndone\n")

[NOTICE] 1 output(s) filtered out

In [4]:
# --- Cell C: download a model and load it. WRITES TO DISK (~a few hundred MB). ---
library(reticulate)
hf <- import("fbri.package_managers.huggingface")

REPO <- "sentence-transformers/all-MiniLM-L6-v2"
MDIR <- file.path(path.expand("~"), "huggingface", REPO, "main")

step <- function(label, expr) {
  t0 <- Sys.time()
  out <- tryCatch(expr, error = function(e)
           paste("ERR:", substr(gsub("[\r\n]+", " ", conditionMessage(e)), 1, 250)))
  cat(sprintf("[%5.1fs] %-22s %s\n", as.numeric(difftime(Sys.time(), t0, units="secs")),
              label, paste(utils::head(as.character(out), 3), collapse=" | ")))
  flush.console(); invisible(out)
}

free0 <- system("df -m ~ | tail -1 | awk '{print $4}'", intern = TRUE)
cat("free MB before:", free0, "\n\n"); flush.console()

# 1. Download. NOTE: hf_download_repo pulls EVERY file in the repo - all format
#    variants (pytorch, onnx, openvino, tf, rust), not just what you need.
step("download", hf$hf_download_repo(REPO))

free1 <- system("df -m ~ | tail -1 | awk '{print $4}'", intern = TRUE)
cat("\nfree MB after: ", free1,
    "  (used ~", as.numeric(free0) - as.numeric(free1), " MB)\n", sep = "")

# 2. Did it land where the source said it would?
cat("path exists:", dir.exists(MDIR), "\n")
cat("files on disk:", length(list.files(MDIR, recursive = TRUE, all.files = TRUE)), "\n")
print(utils::head(list.files(MDIR, recursive = TRUE), 8))
flush.console()

# 3. Load in R. If this HANGS it is trying to reach the hub - interrupt and say so.
tf_ <- import("transformers")
tok <- step("load tokenizer", tf_$AutoTokenizer$from_pretrained(MDIR))
mod <- step("load model",     tf_$AutoModel$from_pretrained(MDIR))

# 4. One forward pass + mean pooling, entirely from R.
if (!is.character(tok) && !is.character(mod)) {
  torch <- import("torch")
  enc <- tok(list("Meta Content Library research"), padding = TRUE,
             truncation = TRUE, return_tensors = "pt")
  out <- step("forward pass", mod(input_ids = enc$input_ids,
                                  attention_mask = enc$attention_mask))
  if (!is.character(out)) {
    m   <- enc$attention_mask$unsqueeze(-1L)$float()
    emb <- torch$sum(out$last_hidden_state * m, dim = 1L) /
           torch$clamp(m$sum(dim = 1L), min = 1e-9)
    cat("embedding shape:", py_str(emb$shape), "\n")
    cat("first 5 values :", paste(round(as.numeric(emb$detach()$numpy())[1:5], 4),
                                  collapse = ", "), "\n")
  }
}
cat("\ndone\n")

[NOTICE] 1 output(s) filtered out

In [5]:
library(reticulate)
hf       <- import("fbri.package_managers.huggingface")
requests <- import("requests")

REPO <- "sentence-transformers/all-MiniLM-L6-v2"

# 1. Ask the proxy for the model's commit metadata (same endpoint hf_list_files uses)
info <- tryCatch({
  r <- requests$get(paste0(hf$HF_ENDPOINT, "/api/models/", REPO, "/revision/main"))
  cat("status:", r$status_code, "\n")
  r$json()
}, error = function(e) paste("ERR:", conditionMessage(e)))

SHA <- tryCatch(if (is.list(info)) info[["sha"]] else NULL, error = function(e) NULL)
if (is.null(SHA)) SHA <- NA_character_
cat("sha for 'main':", SHA, "\n")
cat("keys available:", paste(utils::head(names(info), 12), collapse = ", "), "\n\n")
flush.console()

# 2. Which revision strings does the gate accept?
probe <- function(rev, note) {
  res <- tryCatch(sprintf("OK %4d files", length(unlist(hf$hf_list_files(REPO, rev)))),
           error = function(e) {
             code <- regmatches(conditionMessage(e),
                       regexpr("[0-9]{3} (Client|Server) Error", conditionMessage(e)))
             if (!length(code)) "ERR (other)" else sprintf("ERR %s", code)
           })
  cat(sprintf("  %-14s %-22s %s\n", res, note, substr(rev, 1, 45))); flush.console()
}

probe("main", "control")
if (!is.na(SHA)) {
  probe(SHA,                "full commit SHA")
  probe(substr(SHA, 1, 7),  "short SHA (7 chars)")
}
probe("v1.0",                   "tag, probably absent")
probe("nonexistent-branch-xyz", "definitely absent")

# 3. Does a pinned revision get its OWN directory, as the source implies?
if (!is.na(SHA)) {
  cat("\ndownloading config.json at pinned SHA...\n"); flush.console()
  tryCatch(hf$hf_download_file("config.json", REPO, SHA),
           error = function(e) cat("ERR:", conditionMessage(e), "\n"))
  p <- file.path(path.expand("~"), "huggingface", REPO, SHA, "config.json")
  cat("pinned file exists:", file.exists(p), "\n")
  cat("at:", p, "\n")
  cat("sibling dirs now:",
      paste(list.files(file.path(path.expand("~"), "huggingface", REPO)), collapse = ", "), "\n")
}
cat("\ndone\n")

[NOTICE] 1 output(s) filtered out